In [ ]:
import scanpy

In [ ]:
adata = sc.read("../results/adata/08-preprocess.h5ad")

## Wilcoxin Sign Rank
### Cell Types

In [ ]:
sc.tl.rank_genes_groups(adata, "leiden_0.5", method="wilcoxon")
sc.pl.rank_genes_groups(adata)

In [ ]:
sc.tl.filter_rank_genes_groups(adata, min_in_group_fraction=0.2,
                               max_out_group_fraction=0.2)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata, groupby="leiden_0.5", n_genes=5, key="rank_genes_groups_filtered")

In [ ]:
df = sc.get.rank_genes_groups_df(adata, group=None)
# df.to_csv("output/diff_exp_myel_v_non-myel.csv")

In [ ]:
for group in adata.uns["rank_genes_groups"]["names"].dtype.names:
    display(HTML(f"<h1>{group} vs. rest</h1>"))
    sc.pl.umap(adata, ncols=3, color=[subcluster_key] + adata.uns["rank_genes_groups"]["names"][group][:8].tolist())


### Treatments

## Pseudobulk

In [ ]:
pdata = dc.pp.pseudobulk(
    adata,
    sample_col="sample",
    groups_col="cell_type",
    layer="soupx_rounded",          
    mode="sum",
    
)

In [ ]:
dc.pl.filter_samples(
    adata=pdata,
    groupby=["treatment", "sample", "cell_type"], 
    min_cells=5,
    min_counts=2000,
    figsize=(5, 8)
)

In [ ]:
dc.pp.filter_samples(
    pdata,
    min_cells=5,
    min_counts=500)
pdata.obs[subcluster_key].value_counts()

In [ ]:
with localconverter(ro.default_converter + pandas2ri.converter):
    ro.globalenv["counts"] = pdata.to_df().astype(int).T
    ro.globalenv["meta"] = pdata.obs[["sample", "cell_type", "treatment"]].copy()
    ro.globalenv["outdir"] = "output"

In [ ]:
%%R
suppressMessages(library(DESeq2))

meta$cell_type <- relevel(factor(meta$cell_type), ref = "non-myelinating")

dds <- DESeqDataSetFromMatrix(
    countData=as.matrix(counts), 
    colData=meta, 
    design= ~ cell_type)

dds <- DESeq(dds)

vst <- assay(vst(dds, blind = FALSE))
vst_df <- as.data.frame(vst)

# Dispersion Estimates
png(paste0(outdir,"/dispersions-schwann.png"))
plotDispEsts(dds)
dev.off()


In [ ]:
%%R

resultsNames(dds)
res <- results(dds)
res_df <- as.data.frame(res)
res_df$gene <- rownames(res_df)

In [ ]:
with localconverter(ro.default_converter + pandas2ri.converter):
    vst_df = ro.globalenv["vst_df"]
    res_df = ro.globalenv["res_df"]

In [ ]:
df = df.copy().dropna(subset=["log2FoldChange", "padj"])
    
# Clip padj to avoid log10(0) = -inf
df["-log10_padj"] = -np.log10(df["padj"].clip(lower=1e-300))
   
plt.figure()
plt.scatter(x=df["log2FoldChange"], y=df["-log10_padj"], s=1)
plt.xlabel("$log_{2}$ Fold Change")
plt.ylabel("-logFDR")
plt.axhline(-np.log10(padj_thresh), color="grey", linestyle="--")

top = df.nsmallest(n_labels, "padj")
texts = []
for _, row in top.iterrows():
    texts.append(plt.text(row["log2FoldChange"], row["-log10_padj"], row.name, fontsize=7))
adjust_text(texts, arrowprops=dict(arrowstyle="-", color="grey", lw=0.5))

plt.show()

In [ ]:
vst_top = vst_df.loc[res_df.index[:500].tolist()]
scaler = StandardScaler()
vst_scaled = pd.DataFrame(
    scaler.fit_transform(vst_top.T).T,  # scale across samples
    index=vst_top.index,
    columns=vst_top.columns)

dist = pdist(vst_scaled, metric="correlation")
Z = linkage(dist, method="average")
g = sns.clustermap(
    vst_scaled,
    row_linkage=Z,              # use precomputed linkage
    col_cluster=True,           # also cluster samples
    metric='correlation',       # for column clustering
    method='average',
    # cmap='RdYlBu_r',
    figsize=(14, 10),
    yticklabels=False)
plt.show()